In [0]:
csv_path = "/Volumes/workspace/default/fraudguard_data/PS_20174392719_1491204439457_log.csv"

df_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(csv_path)
)

print(f"Total rows: {df_raw.count():,}")

In [0]:
# Cell: BronzeTableWriter — OOP class jo raw data ko Delta table mein persist karti hai

class BronzeTableWriter:
    """
    Encapsulates Bronze-layer write logic.
    OOP concept: Encapsulation — catalog/schema/table naam sab isi class
    ke andar hain, bahar se koi seedha access nahi karta.
    """

    def __init__(self, catalog: str, schema: str, table_name: str):
        self.catalog = catalog
        self.schema = schema
        self.table_name = table_name

    @property
    def full_table_name(self) -> str:
        return f"{self.catalog}.{self.schema}.{self.table_name}"

    def write_batch(self, df, mode: str = "overwrite") -> None:
        """Poora DataFrame ek Delta table mein likhta hai (source-of-truth)."""
        (
            df.write
            .format("delta")
            .mode(mode)
            .option("overwriteSchema", "true")
            .saveAsTable(self.full_table_name)
        )
        print(f" Wrote data to {self.full_table_name}")


# Apna catalog/schema check kar lena (Catalog tab mein dikha tha upload karte waqt)
bronze_writer = BronzeTableWriter(
    catalog="workspace",
    schema="default",
    table_name="bronze_transactions",
)
from pyspark.sql import functions as F

# Pichla processed step nikalo (agar table khali hai to 0 se shuru)
result = spark.sql(
    "SELECT COALESCE(MAX(last_step_processed), 0) AS s FROM workspace.default.ingestion_state"
).collect()
last_step = result[0]["s"]

# Har run mein sirf agla chunk process karo — 50 "hours" ka data
next_step = last_step + 50
new_chunk = df_raw.filter((F.col("step") > last_step) & (F.col("step") <= next_step))

chunk_count = new_chunk.count()
print(f"Processing steps {last_step+1} to {next_step}: {chunk_count} new rows")

if chunk_count > 0:
    bronze_writer.write_batch(new_chunk, mode="append")

    spark.sql(f"""
        MERGE INTO workspace.default.ingestion_state t
        USING (SELECT 'paysim' AS source_name, {next_step} AS last_step_processed) s
        ON t.source_name = s.source_name
        WHEN MATCHED THEN UPDATE SET t.last_step_processed = s.last_step_processed
        WHEN NOT MATCHED THEN INSERT *
    """)
    print(f" Bronze updated. New last_step_processed = {next_step}")
else:
    print(" Koi naya data nahi mila")

In [0]:
spark.sql(f"SELECT COUNT(*) AS total_rows FROM {bronze_writer.full_table_name}").show()
spark.sql(f"DESCRIBE DETAIL {bronze_writer.full_table_name}").show(truncate=False)